# Ising flow — Julia + Makie

In [ ]:
# Run once to install dependencies.
using Pkg
# Pkg.add(["NPZ", "Interpolations", "Meshing", "GeometryBasics", "LaTeXStrings"])

In [11]:
using NPZ
using Interpolations
using Meshing
using GeometryBasics
using Random
using LaTeXStrings

# =================================================================
# PICK ONE BACKEND — uncomment exactly one block below.
# RESTART THE KERNEL before re-running after a change; Makie backends
# can't cleanly swap in a live session.
# Each block defines `show_fig(fig)` — figure cells call that.
# =================================================================

In [12]:
# # --- A: GLMakie popup (interactive OS window) ---  [DEFAULT, verified]

using GLMakie
GLMakie.activate!()
Makie.inline!(false)
# show_fig(fig) = display(GLMakie.Screen(), fig)
show_fig(fig) = display(fig)

show_fig (generic function with 1 method)

$$\frac{d\hat r}{d\ell}=2\hat r+3\tilde u-3\tilde u\hat r,\qquad
\frac{d\tilde u}{d\ell}=(4-d)\tilde u+10\tilde v-9\tilde u^2,\qquad
\frac{d\tilde v}{d\ell}=(6-2d)\tilde v-45\tilde u\tilde v+27\tilde u^3$$

In [24]:
d = 3.
ϵ = 4 - d

βr(r, u, v) = 2*r - 3*u*r # + 3*u
βu(r, u, v) = (4-d)*u - 9*u^2 # + 10*v 
βv(r, u, v) = (6-2*d)*v - 45*u*v #+ 27*u^3 
# βv(r, u, v) = 0

# Flow on the RG "upward" direction (negative beta).
V(r,u,v) = (βr(r,u,v), βu(r,u,v), βv(r,u,v))

V (generic function with 1 method)

In [25]:
r0, u0, v0 = 0.5, 0.2, 0.2

βv(r0, u0, v0)

-1.8

In [26]:
function simulate(V, r0, u0, v0; dt=0.002, nsteps=1000)
    traj = zeros(nsteps, 3)
    traj[1, :] = [r0, u0, v0]
    for i in 2:nsteps
        r, u, v = traj[i-1, :]
        dr, du, dv = V(r, u, v)
        traj[i, :] = [r + dt*dr, u + dt*du, v + dt*dv]
    end
    return traj
end

function get_init(N, rr, ur, vr; rng=Random.default_rng())
    inits = Vector{NTuple{3,Float64}}()
    for _ in 1:N
        r0 = rand(rng) * (rr[2] - rr[1]) + rr[1]
        u0 = rand(rng) * ur
        v0 = rand(rng) * vr
        push!(inits, (r0, u0, v0))
    end
    return inits
end

get_init (generic function with 1 method)

In [63]:
function plot_flow3D!(ax, N, trajs;
                       nsteps=1000, cmap=:plasma, crange=(-10.0, 10.0),
                       seed=nothing, linewidth=3, subsample=5, ms=10)

    cmin, cmax = Float32(crange[1]), Float32(crange[2])

    # Build one big NaN-separated strip so we get a single lines! draw call.
    # NOTE: NaN goes only in x/y/z (that's what breaks the line); the color array
    # uses a finite sentinel because NaN in per-vertex color can confuse shaders.
    segs = [traj[1:subsample:end, :] for traj in trajs]
    npts = sum(size(s, 1) for s in segs) + length(segs)  # +1 sep per traj

    xs = Vector{Float32}(undef, npts)
    ys = Vector{Float32}(undef, npts)
    zs = Vector{Float32}(undef, npts)
    cs = Vector{Float32}(undef, npts)

    i = 1
    for s in segs
        n = size(s, 1)
        @inbounds for k in 1:n
            xs[i] = s[k, 2]
            ys[i] = s[k, 3]
            zs[i] = s[k, 1]
            cs[i] = clamp(Float32(s[k, 1]), cmin, cmax)
            i += 1
        end
        # Strip separator: NaN in position only.
        xs[i] = NaN32; ys[i] = NaN32; zs[i] = NaN32; cs[i] = cmin
        i += 1
    end

    start_x = Float32[t[1, 2]     for t in trajs]
    start_y = Float32[t[1, 3]     for t in trajs]
    start_z = Float32[t[1, 1]     for t in trajs]
    end_x   = Float32[t[end, 2]   for t in trajs]
    end_y   = Float32[t[end, 3]   for t in trajs]
    end_z   = Float32[t[end, 1]   for t in trajs]
    end_c   = Float32[clamp(Float32(t[end, 1]), cmin, cmax) for t in trajs]

    lines!(ax, xs, ys, zs; color=cs, colormap=cmap, colorrange=crange, linewidth=linewidth,fxaa=true)
    scatter!(ax, start_x, start_y, start_z; color=:blue, markersize=ms)
    # scatter!(ax, end_x, end_y, end_z;color=end_c, colormap=cmap, colorrange=crange, markersize=ms)

    return trajs
end

plot_flow3D! (generic function with 1 method)

In [ ]:
# Per-axis data lengths that come out equal on screen. Axis3 scales the data by
# 2/width per axis for aspect = :equal, by 2/max(width) for :data, and by
# 2/width * a/max(a) for an explicit aspect tuple a — this returns the inverse
# (up to the common factor 2). Dividing by it lands you in the space where the
# axis box is drawn as a cube, which is where round things have to be built for
# them to come out round on screen, whatever the aspect setting is.
function box_units(lims, asp)
    ws = widths(lims)
    if asp === :equal
        return Vec3f(ws[1], ws[2], ws[3])
    elseif asp === :data
        m = maximum(ws)
        return Vec3f(m, m, m)
    elseif asp isa Tuple || asp isa VecTypes{3}
        return Vec3f((ws .* maximum(asp) ./ asp)...)
    else
        error("unsupported Axis3 aspect: $asp")
    end
end
box_units(ax) = box_units(ax.finallimits[], ax.aspect[])

# Unit disc lying in the u–v (:uv) or r–v (:rv) plane: a triangle fan around the
# centre, single winding (Makie doesn't cull back faces, so it stays visible
# from either side, and a single winding keeps the alpha from doubling up).
function disc_mesh(plane::Symbol; n=64)
    θ = range(0, 2π, length=n+1)[1:end-1]
    rim = if plane === :uv
        Point3f[Point3f(cos(t), sin(t), 0) for t in θ]
    elseif plane === :rv
        Point3f[Point3f(0, cos(t), sin(t)) for t in θ]
    else
        error("plane must be :uv or :rv")
    end
    return GeometryBasics.Mesh(vcat([Point3f(0, 0, 0)], rim),
                               [GLTriangleFace(1, i + 1, (i % n) + 2) for i in 1:n])
end

# Projections of the trajectory start points onto two walls of the box: the u–v
# wall (r flattened onto the floor) and the r–v wall (u flattened onto the
# side). The v = 0 wall is left alone on purpose — that one carries the
# streamplot.
#
# The markers are flat discs *lying in* the wall planes, not camera-facing dots,
# so they foreshorten with the wall like a real projection. `radius` and `pad`
# are in box_units, i.e. fractions of the on-screen box, so the discs stay round
# for any aspect setting.
#
# Call this *after* xlims!/ylims!/zlims!: the wall positions and the disc shape
# are read off ax.finallimits once, and that only holds the right values after
# the limits are set. Override u_plane / r_plane to pin the walls elsewhere.
function plot_imprints!(ax, trajs;
                        u_plane=nothing, r_plane=nothing,
                        color=(:blue, 0.35), radius=0.02, pad=0.005)

    lo = minimum(ax.finallimits[])
    w  = box_units(ax)
    u_wall = isnothing(u_plane) ? lo[1] : u_plane
    r_wall = isnothing(r_plane) ? lo[3] : r_plane

    start_u = Float32[t[1, 2] for t in trajs]
    start_v = Float32[t[1, 3] for t in trajs]
    start_r = Float32[t[1, 1] for t in trajs]
    n = length(trajs)

    # u–v wall: keep (u, v), flatten r.
    meshscatter!(ax, start_u, start_v, fill(Float32(r_wall + pad * w[3]), n);
                 marker=disc_mesh(:uv),
                 markersize=Vec3f(radius * w[1], radius * w[2], 1),
                 color=color, shading=NoShading, transparency=true, fxaa=true)
    # r–v wall: keep (v, r), flatten u.
    meshscatter!(ax, fill(Float32(u_wall + pad * w[1]), n), start_v, start_r;
                 marker=disc_mesh(:rv),
                 markersize=Vec3f(1, radius * w[2], radius * w[3]),
                 color=color, shading=NoShading, transparency=true, fxaa=true)

    return ax
end

In [ ]:
# Cone arrow heads at the end of the 3D trajectories — same construction as
# add_arrows in papers/2026_tuning_alpha/NRCH_twoloop_julia.ipynb: a ring of
# triangles fanning out to a tip, pointing along the last integration step.
#
# The head is built in box_units (the space where the axis box is a cube) and
# scaled back to data units, so its base is a circle *on screen* rather than in
# data units — otherwise it comes out stretched, badly so here where the r range
# is ~10x the v range. head_len / head_radius / head_shift are therefore
# fractions of the on-screen box; the defaults are the NRCH numbers divided by
# that notebook's box width (2.4), so the heads read the same.
#
# The geometry is rebuilt whenever the limits or the aspect change, so it does
# not matter whether this runs before or after xlims!/ylims!/zlims! (e.g. from
# inside plot_flow3D!, where the limits are not set yet).
#
# `color` takes one colour or one per trajectory. `indx` picks the (x, y, z)
# columns out of a trajectory row: rows are (r, u, v) and the axis is (u, v, r),
# hence (2, 3, 1).
function add_arrows!(ax, trajs;
                     indx=(2, 3, 1), color=:black,
                     head_len=0.075, head_radius=0.033, head_shift=-0.04,
                     n_sides=12)

    cols = color isa AbstractVector ? collect(color) : fill(color, length(trajs))

    _len(v)      = sqrt(v[1]^2 + v[2]^2 + v[3]^2)
    _cross(a, b) = Vec3f(a[2]*b[3] - a[3]*b[2], a[3]*b[1] - a[1]*b[3], a[1]*b[2] - a[2]*b[1])

    for (k, t) in enumerate(trajs)
        size(t, 1) >= 2 || continue
        p_prev = Vec3f(t[end-1, indx[1]], t[end-1, indx[2]], t[end-1, indx[3]])
        p_curr = Vec3f(t[end,   indx[1]], t[end,   indx[2]], t[end,   indx[3]])
        _len(p_curr .- p_prev) > 1e-12 || continue

        head = lift(ax.finallimits, ax.aspect) do lims, asp
            w = box_units(lims, asp)

            # Everything from here on is in box units; the .* w at the end puts
            # the head back into data coordinates.
            d = (p_curr .- p_prev) ./ w
            u = d ./ _len(d)

            # Two directions orthogonal to u, to sweep the ring with.
            ref = abs(u[3]) > 0.9 ? Vec3f(1, 0, 0) : Vec3f(0, 0, 1)
            n = _cross(u, ref); n = n ./ _len(n)
            b = _cross(u, n)

            tip  = p_curr ./ w .- head_shift .* u   # head_shift < 0 pushes it past the end
            base = tip .- head_len .* u
            rim  = [Point3f((base .+ head_radius .* (cos(θ) .* n .+ sin(θ) .* b)) .* w)
                    for θ in range(0, 2π; length=n_sides + 1)[1:end-1]]

            GeometryBasics.Mesh(vcat([Point3f(tip .* w)], rim),
                                [GLTriangleFace(1, i + 1, (i % n_sides) + 2) for i in 1:n_sides])
        end

        mesh!(ax, head; color=cols[k], shading=NoShading, fxaa=true)
    end

    return ax
end

In [66]:
# Flat arrow head for the streamplot: base spanning x ∈ [-½, ½] at z = 0, tip at
# z = 1 (the marker convention: +z is the arrow direction). Both windings are
# included so it stays visible from either side. Since the flow lies in the
# v = 0 plane the head ends up flat in that plane too, which reads much better
# than Makie's default 3D cone seen edge-on.
arrow_head_2d() = GeometryBasics.Mesh(
    # Point3f[(-0.5, 0, 0), (0.5, 0, 0), (0, 0, 1)],
    # GLTriangleFace[(1, 2, 3), (3, 2, 1)]
    Point3f[(-1, 0, 0), (1, 0, 0), (0, 0, 1)],
    GLTriangleFace[(1, 2, 3), (3, 2, 1)]
    )

# Streamlines of (βu, βr) on the v = 0 plane, drawn into the 3D axis.
# streamplot! needs a 3D field on an Axis3, so the v-component is zeroed and the
# v-interval is degenerate — every streamline then stays exactly at v = 0.
# NOTE: v = 0 is not invariant (βv = 27u³ there); this is the projected (u, r)
# flow in that slice, not a sub-flow of the full system.
#
# Arrow knobs, all in data units (u/r units, *not* pixels):
#   arrow_width, arrow_length  — size of the head; start here when tuning.
#   arrow_color                — colour of the heads, independent of linecolor.
#   arrow_head                 — swap in `Makie.automatic` for the default cone,
#                                or any mesh/marker.
#   gridsize, density          — how many streamlines, hence how many arrows.
# arrow_size=automatic is useless here: Makie scales it by the smallest box
# width, which is the degenerate v-interval, so the arrows come out invisible.
function plot_stream_v0!(ax, urange, rrange;
                         gridsize=18, density=2.0, stepsize=0.001, maxsteps=2000,
                         linecolor=:gray55, linewidth=2,
                         arrow_color=linecolor,
                         arrow_width=0.1, arrow_length=0.2,
                         arrow_head=arrow_head_2d())

    # streamplot passes points as (x, y, z) = (u, v, r), and wants the same back.
    f(p) = Point3f(βu(p[3], p[1], 0.0), 0.0, βr(p[3], p[1], 0.0))

    sp = streamplot!(ax, f, urange[1]..urange[2], -1.0f-6..1.0f-6, rrange[1]..rrange[2];
                gridsize=(gridsize, 1, gridsize), density=density,
                stepsize=stepsize, maxsteps=maxsteps,
                color=p -> linecolor, linewidth=linewidth,
                arrow_head=arrow_head,
                arrow_size=Vec3f(0))  # heads hidden — redrawn below

    # Makie derives the arrow colour from the same `color` function as the
    # lines, so there is no attribute to colour them separately. Instead the
    # recipe's own heads are scaled to zero and redrawn here from its computed
    # positions/directions, which lets `arrow_color` be anything. NoShading
    # keeps the flat head at exactly that colour instead of letting the light
    # darken the triangle (which is what the built-in heads do).
    meshscatter!(ax, sp.arrow_positions[];
                 rotation=sp.arrow_directions[],
                 marker=arrow_head,
                 markersize=Vec3f(arrow_width, arrow_width, arrow_length),
                 color=arrow_color, shading=NoShading, fxaa=true)

    return sp
end

plot_stream_v0! (generic function with 1 method)

In [67]:
yellow      = colorant"#fff800"
pink        = colorant"#ffdcfe"
pink        = colorant"#ffdcfe"
pink        = colorant"#f03ed4"
blue        = colorant"#a3f8ff"
green       = colorant"#c8ffce"
turquise    = colorant"#41e3c0"
red         = RGBf(0.8,0.2,0.2)

# cmap = cgrad([blue, pink, green])

In [68]:
cmap = cgrad([blue, blue, pink, green, green], [0.0, 0.1, .5, .9, 1.0])
cmap = cgrad([:black, :black],[0, 1])

In [81]:
set_theme!(theme_latexfonts())

rr = (-0.3, 0.3)
ur = 0.2
vr = 0.1

ms = 20
lw = 5
s = 5
fs = 40

N = 100

function get_init(N, rr, ur, vr; rng=Random.default_rng())
    return [
        # (-0.28, 0.08, 0.08),
        # (-0.31, 0.08, 0.08),
        # (-0.145, 0.005, 0.03),
        (+0.05, 0.048, 0.078),
        (-0.1, 0.235, 0.08),
        (0.0, 0.1, 0.095),
    ]
end

N = get_init(0,0,0,0);

nsteps = [500, 500, 300]

rng = Random.MersenneTwister(1)
inits = get_init(N, rr, ur, vr; rng=rng)
trajs = [simulate(V, r0, u0, v0; nsteps=nsteps[i]) for (i, (r0, u0, v0)) in enumerate(inits)];

In [82]:
# a b c


fig = Figure(size=(150*s, 100*s); fontsize = 30)
ax  = Axis3(fig[1, 1]; 
    xlabel=L"u", ylabel=L"v", zlabel=L"r", viewmode = :fit,aspect = :equal,
    xlabelsize = fs, ylabelsize = fs, zlabelsize = fs, 
    xlabeloffset = 50, ylabeloffset = 50, zlabeloffset = 70,
    )

dr = (rr[2] - rr[1]) / 2

# Streamlines of the (u, r) flow on the v = 0 plane, under the trajectories.
plot_stream_v0!(ax, (0, 1.2*ur), (rr[1] - dr, rr[2] + dr);
    gridsize     = 18,      # more streamlines -> mor arroews
    linecolor    = :grey,
    arrow_color  = :grey,   # heads, independent of the lines
    linewidth    = 2,
    arrow_width  = 0.005,   # data units
    arrow_length = 0.02,    # data units
    density      = 0.8,
    )

 # Plot the flow trajectories.
trajs = plot_flow3D!(ax, N, trajs; linewidth=lw, ms=ms, cmap=cmap)

xlims!(ax, (0, 1.2*ur))
ylims!(ax, (0, max(0.01,vr)))
zlims!(ax, (rr[1] - dr, rr[2] + dr))

# Start points projected onto the u–v and r–v walls (needs the limits above).
plot_imprints!(ax, trajs;
    radius = 0.02,          # disc radius, in fractions of the box width
    color  = (:blue, 0.35),
    )

# Cone heads at the end of the trajectories (needs the limits above too).
add_arrows!(ax, trajs;
    color       = :black,
    head_len    = 0.075,    # fractions of the box width
    head_radius = 0.033,
    )

show_fig(fig)

GLMakie.Screen(...)

In [83]:
save("ising_flow.png", fig; px_per_unit = 4)